# 🤖 NanoChat - GPU Inference

Run your 162M parameter NanoChat model on Colab's free GPU!

**Instructions:**
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Run all cells
3. Upload your `nanochat_large_final.pkl` when prompted

In [ ]:
# Install dependencies
!pip install -q jax[cuda12] tiktoken gradio

In [ ]:
# Check GPU
import jax
print("Devices:", jax.devices())
print("GPU available!" if 'gpu' in str(jax.devices()[0]).lower() else "WARNING: No GPU - go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Upload your model
from google.colab import files
print("Upload your nanochat_large_final.pkl file:")
uploaded = files.upload()

In [ ]:
# Load model
import pickle
import jax.numpy as jnp
import numpy as np
import tiktoken

print("Loading model...")
with open('nanochat_large_final.pkl', 'rb') as f:
    ckpt = pickle.load(f)

params = jax.tree.map(jnp.array, ckpt['params'])
cfg = ckpt['config']
enc = tiktoken.get_encoding("gpt2")

print(f"✓ Loaded model: {cfg}")

In [ ]:
# Model functions
def rms_norm(x, gamma, eps=1e-6):
    ms = jnp.mean(jnp.square(x.astype(jnp.float32)), axis=-1, keepdims=True)
    return (x.astype(jnp.float32) * jax.lax.rsqrt(ms + eps) * gamma).astype(x.dtype)

def apply_rope(xq, xk, cos, sin):
    def rotate_half(x):
        x1, x2 = jnp.split(x, 2, axis=-1)
        return jnp.concatenate([-x2, x1], axis=-1)
    cos = cos[None, None, :, :].astype(xq.dtype)
    sin = sin[None, None, :, :].astype(xq.dtype)
    return (xq * cos) + (rotate_half(xq) * sin), (xk * cos) + (rotate_half(xk) * sin)

def self_attention(x, attn_params, n_head, mask, cos, sin):
    batch, seq_len, n_embd = x.shape
    head_dim = n_embd // n_head
    q = jnp.dot(x, attn_params['wq']).reshape(batch, seq_len, n_head, head_dim).transpose(0, 2, 1, 3)
    k = jnp.dot(x, attn_params['wk']).reshape(batch, seq_len, n_head, head_dim).transpose(0, 2, 1, 3)
    v = jnp.dot(x, attn_params['wv']).reshape(batch, seq_len, n_head, head_dim).transpose(0, 2, 1, 3)
    q, k = apply_rope(q, k, cos[:seq_len], sin[:seq_len])
    scores = jnp.matmul(q, jnp.swapaxes(k, -2, -1)) / jnp.sqrt(head_dim)
    scores = scores + mask[:seq_len, :seq_len]
    weights = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(x.dtype)
    output = jnp.matmul(weights, v).transpose(0, 2, 1, 3).reshape(batch, seq_len, -1)
    return jnp.dot(output, attn_params['wo'])

def transformer_block(x, block_params, n_head, mask, cos, sin):
    h = rms_norm(x, block_params['norm1'])
    x = x + self_attention(h, block_params['attn'], n_head, mask, cos, sin)
    h = rms_norm(x, block_params['norm2'])
    x = x + jnp.dot(jax.nn.gelu(jnp.dot(h, block_params['mlp']['w1'])), block_params['mlp']['w2'])
    return x

def forward(params, tokens, n_head, mask, cos, sin):
    x = params['token_emb'][tokens].astype(jnp.bfloat16)
    for block in params['blocks']:
        x = transformer_block(x, block, n_head, mask, cos, sin)
    x = rms_norm(x, params['final_norm'])
    return jnp.dot(x, params['output_head']).astype(jnp.float32)

# Precompute
block_size = cfg['block_size']
n_head = cfg['n_head']
n_embd = cfg['n_embd']
head_dim = n_embd // n_head

mask = jnp.where(jnp.tril(jnp.ones((block_size, block_size))), 0.0, -jnp.inf)
dims = jnp.arange(0, head_dim, 2)
freqs = 1.0 / (10000.0 ** (dims / head_dim))
pos = jnp.arange(block_size)
angles = jnp.outer(pos, freqs)
angles = jnp.repeat(angles, 2, axis=-1)
cos, sin = jnp.cos(angles), jnp.sin(angles)

print("✓ Model ready!")

In [ ]:
# Generate function
@jax.jit
def forward_jit(params, tokens):
    return forward(params, tokens, n_head, mask, cos, sin)

def generate(prompt, max_tokens=100, temperature=0.7):
    tokens = enc.encode(prompt)
    key = jax.random.PRNGKey(np.random.randint(0, 10000))
    
    for _ in range(max_tokens):
        if len(tokens) >= block_size:
            tokens = tokens[-block_size+1:]
        
        logits = forward_jit(params, jnp.array([tokens]))
        logits = logits[0, -1, :] / temperature
        
        # Top-p sampling
        probs = jax.nn.softmax(logits)
        sorted_indices = jnp.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        cumsum = jnp.cumsum(sorted_probs)
        cutoff = jnp.searchsorted(cumsum, 0.9) + 1
        top_probs = sorted_probs[:cutoff]
        top_probs = top_probs / top_probs.sum()
        
        key, subkey = jax.random.split(key)
        idx = jax.random.choice(subkey, cutoff, p=top_probs)
        next_token = int(sorted_indices[idx])
        tokens.append(next_token)
        
        # Stop conditions
        decoded = enc.decode(tokens)
        if "User:" in decoded.split("Assistant:")[-1]:
            break
    
    return enc.decode(tokens)

# Warmup
print("Warming up (first run is slow)...")
_ = generate("User: Hi\nAssistant:", max_tokens=5)
print("✓ Ready to chat!")

In [ ]:
# Chat function
import time

def chat(message, max_tokens=100, temperature=0.7):
    prompt = f"User: {message}\nAssistant:"
    
    start = time.time()
    output = generate(prompt, max_tokens=max_tokens, temperature=temperature)
    elapsed = time.time() - start
    
    # Extract response
    if "Assistant:" in output:
        response = output.split("Assistant:")[-1]
        if "User:" in response:
            response = response.split("User:")[0]
        response = response.strip()
    else:
        response = output
    
    print(f"\n🤖 NanoChat: {response}")
    print(f"\n[{elapsed:.2f}s]")
    return response

In [ ]:
# Try it!
chat("What is photosynthesis?")

In [ ]:
# Ask your own question!
chat("Explain how computers work")

In [ ]:
# Interactive chat loop
print("=" * 50)
print("NanoChat - Interactive Mode")
print("Type 'quit' to exit")
print("=" * 50)

while True:
    user_input = input("\nYou: ")
    if user_input.lower() == 'quit':
        print("Goodbye!")
        break
    chat(user_input)

# Chat UI

Run this cell to launch a web interface. You'll get a **public link** you can share with anyone!

In [ ]:
# Gradio Chat UI
import gradio as gr

def respond(message, history, temperature, max_tokens):
    prompt = f"User: {message}\nAssistant:"
    output = generate(prompt, max_tokens=int(max_tokens), temperature=temperature)
    
    if "Assistant:" in output:
        response = output.split("Assistant:")[-1]
        if "User:" in response:
            response = response.split("User:")[0]
        response = response.strip()
    else:
        response = output
    
    return response

demo = gr.ChatInterface(
    respond,
    title="NanoChat",
    description="Chat with your 162M parameter model",
    additional_inputs=[
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(10, 200, value=100, step=10, label="Max Tokens"),
    ],
    examples=[
        ["What is machine learning?"],
        ["Explain photosynthesis"],
        ["Write a short poem about coding"],
    ],
    theme="soft",
)

demo.launch(share=True)